In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import resample

In [2]:
df = pd.read_csv("earthquake_week1_cleaned.csv")

print("Dataset Loaded")
print("Shape:", df.shape)

Dataset Loaded
Shape: (1300, 6)


In [3]:
df.info()
df.describe()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1300 entries, 0 to 1299
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   magnitude  1300 non-null   float64
 1   depth      1300 non-null   int64  
 2   cdi        1300 non-null   int64  
 3   mmi        1300 non-null   int64  
 4   sig        1300 non-null   int64  
 5   alert      1300 non-null   object 
dtypes: float64(1), int64(4), object(1)
memory usage: 61.1+ KB


Index(['magnitude', 'depth', 'cdi', 'mmi', 'sig', 'alert'], dtype='object')

In [4]:
df.isnull().sum()

magnitude    0
depth        0
cdi          0
mmi          0
sig          0
alert        0
dtype: int64

In [5]:
print("Duplicates:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("Duplicates Removed")

Duplicates: 44
Duplicates Removed


In [6]:
numeric_cols = ['magnitude','depth','cdi','mmi','sig']

Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

df = df[~((df[numeric_cols] < (Q1 - 1.5 * IQR)) |
          (df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

print("Outliers Removed")

Outliers Removed


In [7]:
df[numeric_cols] = df[numeric_cols].astype(float)

In [8]:
le = LabelEncoder()
df['alert'] = le.fit_transform(df['alert'])

print("Encoding Completed")

Encoding Completed


In [9]:
# green → 0
# yellow → 1
# orange → 2
# red → 3

In [10]:
scaler = StandardScaler()

df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print("Feature Scaling Done")

Feature Scaling Done


In [11]:
df['alert'].value_counts()

alert
2    299
1    294
3    248
0    170
Name: count, dtype: int64

In [12]:
majority = df[df.alert == df.alert.mode()[0]]
minority = df[df.alert != df.alert.mode()[0]]

minority_upsampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

df = pd.concat([majority, minority_upsampled])

In [13]:
df.to_csv("earthquake_week2_processed.csv", index=False)

print("Week 2 Completed Successfully")

Week 2 Completed Successfully
